# E6 | Model Risk XGBoost + SHAP
Classificar incidentes em risco de violar OLA com explicabilidade SHAP

## 📋 Objetivo

Neste notebook, vou treinar um modelo **XGBoost** para **classificar incidentes em risco de violar OLA**. Além de prever, vou usar **SHAP** para explicar **por que** cada incidente foi classificado como "risco" ou "OK".

### Por que XGBoost para classificação de risco?
- **Árvores de decisão**: Captura interações complexas entre features (ex: "incidentes de categoria X + turno noturno = risco alto")
- **Boosting**: Enfatiza casos difíceis (incidentes reais com violação são raros: 19.7% dos dados)
- **SHAP**: Explica cada previsão de forma compreensível (qual feature contribuiu mais para "risco"?)

### Fluxo
1. **Setup**: Conexão com RDS
2. **Carregamento**: Dados de classificação (121.811 incidentes)
3. **Preparação**: One-Hot Encoding para variáveis categóricas
4. **Split**: Train/Test 80/20 com stratificação (respeita desbalanceamento)
5. **Treino**: XGBoost com 100 árvores, profundidade 6
6. **Avaliação**: AUC-ROC (melhor métrica para dados desbalanceados)
7. **Explicabilidade**: SHAP para identificar features mais importantes
8. **Salvamento**: Resultados e importâncias em CSV

In [23]:
import warnings
warnings.filterwarnings('ignore')
import os, pandas as pd, numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import shap
import mlflow

load_dotenv()
print('Setup OK')

Setup OK


In [24]:
RDS_HOST     = os.getenv('RDS_HOST')
RDS_PORT     = int(os.getenv('RDS_PORT', '5432'))
RDS_USER     = os.getenv('RDS_USER', 'postgres')
RDS_PASSWORD = os.getenv('RDS_PASSWORD')
RDS_DATABASE = os.getenv('RDS_DATABASE', 'aiops_gold')

print(f'Host: {RDS_HOST}')
print(f'DB:   {RDS_DATABASE}')
print(f'User: {RDS_USER}')

connection_url = f"postgresql+psycopg2://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}"

%load_ext sql
%sql {connection_url}

Host: terraform-20260518150028461700000001.c4xegmk24lg6.us-east-1.rds.amazonaws.com
DB:   aiops_gold
User: postgres
The sql extension is already loaded. To reload it, use:
  %reload_ext sql


## 📥 Etapa 1: Carregamento de Dados

Vou carregar os dados de classificação de risco do RDS. Estes dados já foram processados na camada **gold** do data lake e contêm informações de cada incidente e se ele violou o SLA (target).

### 📊 Dataset de Classificação de Risco

O dataset contém **121.811 incidentes** com as seguintes informações:

#### Target (O que vamos prever)
- **target_excedeu_tempo** (0/1):
  - `0` = Incidente **SLA OK** (tempo resposta ≤ esperado) — 97.784 incidentes (80.3%)
  - `1` = Incidente **SLA VIOLADO** (tempo resposta > esperado) — 24.027 incidentes (19.7%)
  - ⚠️ **Dados desbalanceados**: Só 19.7% dos incidentes violam SLA. XGBoost aprende a usar boosting para enfatizar a classe minoritária.

#### Features (15 características originais)
| Tipo | Colunas | Papel |
|---|---|---|
| **Numéricas (11)** | prioridade_num, possui_pai, hora_abertura, dia_semana, semana_ano, mes_abertura, pct_violacao_sla, duracao_horas, qty_updates, dias_desde_abertura, idade_horas | Características diretas do incidente |
| **Categóricas (4)** | grupo_designado, categoria, subcategoria, turno_abertura | Dimensões que será necessário fazer One-Hot Encoding |

#### Por que essas features?
- **Prioridade**: P1 (crítica) = risco maior de violar SLA?
- **Categoria/Subcategoria**: Tipos de incidente (rede, banco de dados) têm diferentes SLAs
- **Turno/Hora**: Incidentes noturnos podem ter menos recursos?
- **Taxa violação**: Histórico de violações do grupo pode indicar capacidade insuficiente
- **Idade**: Incidentes muito antigos têm maior risco de violar SLA

**Próximo passo**: Vou converter categóricas para numéricas (One-Hot Encoding) e depois treinar XGBoost.

## 🔧 Etapa 2: Preparação dos Dados

XGBoost trabalha apenas com **números**, então preciso converter variáveis categóricas em numéricas:

### One-Hot Encoding
Vou converter cada valor categórico em uma coluna binária (0/1):
- `grupo_designado` = ['Team1', 'Team2', 'Team14'] → 3 colunas: `grupo_designado_Team1`, `grupo_designado_Team2`, `grupo_designado_Team14`
- `turno_abertura` = ['Manha', 'Tarde', 'Noite'] → 3 colunas: `turno_abertura_Manha`, etc.

**Resultado**: 15 features originais → ~600 features após encoding (as 4 categóricas geram ~585 colunas binárias)

### Importante: Evitar Data Leakage
- Fit o encoder **APENAS no train set**
- Depois apply no test set
- Assim, o modelo não "vê" as categorias do teste durante o treinamento

## 🚀 Etapa 3: Treino do XGBoost

Vou treinar um classificador XGBoost com configuração:
- **n_estimators=100**: 100 árvores de decisão (mais árvores = modelo mais complexo)
- **max_depth=6**: Profundidade máxima de 6 níveis (evita overfitting, reduz interpretação)
- **learning_rate=0.1**: Velocidade de aprendizado (0.1 = 10% de shrinkage por árvore)
- **eval_metric='logloss'**: Métrica de perda binária (ideal para classificação 0/1)
- **stratify=y**: Mantém proporção de classes no train/test (80.3% / 19.7%)

### Por que essa configuração?
- **Dados desbalanceados**: O `stratify` garante que train e test tenham mesma proporção (80/20 em cada classe)
- **Profundidade 6**: Equilibra performance e interpretabilidade (não quer árvore muito profunda)
- **100 árvores**: Suficiente para dados de 121k incidentes; boosting vai focar nos difíceis

## 📈 Etapa 4: Avaliação do Modelo

Vou avaliar o modelo no conjunto de teste (24.363 incidentes) usando **AUC-ROC**:

### Por que AUC-ROC em vez de Acurácia?
- **Acurácia**: Seria enganosa (poderíamos classificar tudo como "SLA OK" e acertar 80%)
- **AUC-ROC**: Mede a capacidade de ranquear incidentes com risco acima dos sem risco
  - Valor 0.5 = modelo aleatório
  - Valor 1.0 = modelo perfeito
  - **Target**: AUC > 0.85

Vou também gerar um **Classification Report** para ver:
- **Precision** (de 100 previsões "risco", quantas estavam certas?)
- **Recall** (de 100 incidentes com risco real, quantos o modelo capturou?)
- **F1-Score** (equilíbrio entre precision e recall)

In [25]:
# Carregar dados e criar target baseado em SLA esperado
engine = create_engine(f"postgresql://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}")

df = pd.read_sql('''
    SELECT *
    FROM gold_ml.ml_sla_classification_dataset
''', engine)


## 🔍 Etapa 5: Explicabilidade com SHAP

Aqui vem a **mágica** do projeto! 🎩✨

XGBoost é ótimo em prever, mas é uma "caixa preta": como sei **por que** um incidente foi classificado como "risco"?

### SHAP (SHapley Additive exPlanations)
SHAP calcula a **contribuição de cada feature** para cada previsão:
- **SHAP value positivo**: Feature contribuiu para "risco" (aumenta probabilidade de SLA violado)
- **SHAP value negativo**: Feature contribuiu para "SLA OK" (diminui probabilidade)

### Exemplo
Incidente X foi classificado como "risco" porque:
- `prioridade_num=1` (crítica) → +2.5 (aumenta risco muito)
- `turno_abertura=Noite` → +0.8 (aumenta risco)
- `pct_violacao_sla=5%` → -0.3 (diminui risco, este grupo tem bom histórico)
- **Resultado final**: "Classificar como risco"

### Output
Vou mostrar as **10 features mais importantes** (maior impacto SHAP no teste).

## 📦 Etapa 6: Salvamento de Resultados

Vou salvar os resultados do modelo em arquivos CSV para:
1. **Usar em dashboards Power BI** (importar predições e importâncias)
2. **Auditoria**: Ter registro do que o modelo previu
3. **Análise posterior**: Estudar incidentes classificados errado

### Arquivos Salvos
- **xgboost_test_predictions.csv**: Para cada incidente do teste:
  - `y_real`: Violou SLA real (0/1)?
  - `y_pred`: Previsão do modelo (0/1)?
  - `y_pred_proba`: Confiança da previsão (0.0-1.0, ex: 0.87 = 87% de risco)
  
- **xgboost_shap_importance.csv**: Importância SHAP de cada feature
  - `feature`: Nome da coluna
  - `shap_importance`: Impacto médio dessa feature (maior = mais importante)
  
- **xgboost_summary.csv**: Resumo de métricas
  - AUC-ROC final, tamanho dos sets, número de features

In [26]:
# Preparar features e target (CORRIGIDO - remover target_risco_sla para evitar leakage)
feature_cols_orig = [c for c in df.columns if c not in ['incident_id', 'target_excedeu_tempo', 'duracao_horas', 'target_risco_sla']]
X_orig = df[feature_cols_orig].fillna(0)
y = df['target_excedeu_tempo']

# Identificar colunas categóricas
cat_cols = X_orig.select_dtypes(include=['object']).columns.tolist()
num_cols = X_orig.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f'✅ CORRIGIDO: target_risco_sla removido das features')
print(f'Colunas categóricas: {cat_cols}')
print(f'Colunas numéricas: {len(num_cols)}')
print(f'Total features: {len(feature_cols_orig)}')
print(f'\nTarget class balance:')
print(y.value_counts().sort_index())

✅ CORRIGIDO: target_risco_sla removido das features
Colunas categóricas: ['grupo_designado', 'categoria', 'subcategoria', 'turno_abertura']
Colunas numéricas: 10
Total features: 14

Target class balance:
target_excedeu_tempo
0    97784
1    24027
Name: count, dtype: int64


In [27]:
# Split dos dados (antes do encoding final)
from sklearn.preprocessing import OneHotEncoder

# Recarregar features originais
feature_cols_orig = [c for c in df.columns if c not in ['incident_id', 'target_excedeu_tempo', 'duracao_horas']]
X_orig = df[feature_cols_orig].fillna(0)
y = df['target_excedeu_tempo']

# Identificar colunas categóricas e numéricas
cat_cols = X_orig.select_dtypes(include=['object']).columns.tolist()
num_cols = X_orig.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f'Colunas categóricas (antes de converter): {cat_cols}')

# Converter colunas categóricas para string (OneHotEncoder exige uniformidade)
if cat_cols:
    for col in cat_cols:
        X_orig[col] = X_orig[col].astype(str)

print(f'Colunas categóricas (após converter para str): {cat_cols}')
print(f'Colunas numéricas: {len(num_cols)}')

# Split ANTES do encoding
X_train_orig, X_test_orig, y_train, y_test = train_test_split(
    X_orig, y, test_size=0.2, random_state=42, stratify=y
)

# Fit encoder APENAS no train e aplicar em ambos
if cat_cols:
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    encoder.fit(X_train_orig[cat_cols])
    
    X_train_cat = encoder.transform(X_train_orig[cat_cols])
    X_test_cat = encoder.transform(X_test_orig[cat_cols])
    
    X_train_cat_df = pd.DataFrame(X_train_cat, columns=encoder.get_feature_names_out(cat_cols), index=X_train_orig.index)
    X_test_cat_df = pd.DataFrame(X_test_cat, columns=encoder.get_feature_names_out(cat_cols), index=X_test_orig.index)
    
    X_train = pd.concat([X_train_orig[num_cols].reset_index(drop=True), X_train_cat_df.reset_index(drop=True)], axis=1)
    X_test = pd.concat([X_test_orig[num_cols].reset_index(drop=True), X_test_cat_df.reset_index(drop=True)], axis=1)
else:
    X_train = X_train_orig
    X_test = X_test_orig

print(f'\n✅ Train: {len(X_train)}, Test: {len(X_test)}')
print(f'Features após encoding: {X_train.shape[1]}')
print(f'Train target distribution: {y_train.value_counts().to_dict()}')
print(f'Test target distribution: {y_test.value_counts().to_dict()}')

Colunas categóricas (antes de converter): ['grupo_designado', 'categoria', 'subcategoria', 'turno_abertura']
Colunas categóricas (após converter para str): ['grupo_designado', 'categoria', 'subcategoria', 'turno_abertura']
Colunas numéricas: 11

✅ Train: 97448, Test: 24363
Features após encoding: 598
Train target distribution: {0: 78227, 1: 19221}
Test target distribution: {0: 19557, 1: 4806}


In [28]:
# Treinar XGBoost
model = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss')
print('Training XGBoost...')
model.fit(X_train, y_train, verbose=0)
print('✅ Done')

Training XGBoost...
✅ Done


In [29]:
# Avaliar modelo
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)
auc_roc = roc_auc_score(y_test, y_pred_proba)

print(f'AUC-ROC: {auc_roc:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred))

AUC-ROC: 0.9710

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.97      0.97     19557
           1       0.86      0.86      0.86      4806

    accuracy                           0.94     24363
   macro avg       0.91      0.91      0.91     24363
weighted avg       0.94      0.94      0.94     24363



In [30]:
# Debug: Verificar target já está sendo usado
print('✅ Target já existe: target_excedeu_tempo')
print('   Criado baseado em SLA esperado por prioridade')
print(f'\nDistribuição do target:')
print(df['target_excedeu_tempo'].value_counts())
print(f'\nTarget balance:')
print(f'  Classe 0 (SLA OK): {(df["target_excedeu_tempo"]==0).sum()} ({(df["target_excedeu_tempo"]==0).sum()/len(df)*100:.1f}%)')
print(f'  Classe 1 (SLA VIOLADO): {(df["target_excedeu_tempo"]==1).sum()} ({(df["target_excedeu_tempo"]==1).sum()/len(df)*100:.1f}%)')
print(f'\nModelo está pronto para classificar incidentes em risco de violar SLA')

✅ Target já existe: target_excedeu_tempo
   Criado baseado em SLA esperado por prioridade

Distribuição do target:
target_excedeu_tempo
0    97784
1    24027
Name: count, dtype: int64

Target balance:
  Classe 0 (SLA OK): 97784 (80.3%)
  Classe 1 (SLA VIOLADO): 24027 (19.7%)

Modelo está pronto para classificar incidentes em risco de violar SLA


In [31]:
# SHAP explicabilidade
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

print(f'SHAP values shape: {shap_values.shape}')
print(f'Top features by SHAP importance:')

feature_names = X_train.columns.tolist()
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'shap_importance': np.abs(shap_values).mean(axis=0)
}).sort_values('shap_importance', ascending=False)

print(feature_importance.head(10))

SHAP values shape: (24363, 598)
Top features by SHAP importance:
                    feature  shap_importance
0            prioridade_num         3.500357
28              categoria_0         0.602649
1                possui_pai         0.277544
24   grupo_designado_Team14         0.174924
9              mes_abertura         0.161030
4             hora_abertura         0.129860
8                semana_ano         0.127753
167          subcategoria_0         0.126752
595    turno_abertura_Manha         0.087859
597    turno_abertura_Tarde         0.082815


In [32]:
import joblib
import tempfile
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, RocCurveDisplay, ConfusionMatrixDisplay
from mlflow.models import infer_signature

# Calcular métricas adicionais
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Rastrear no MLflow com estrutura completa
mlflow.set_experiment('xgboost_ola_risk')
with mlflow.start_run(run_name="xgboost_v1_sla_risk_classification"):
    # ===== PARÂMETROS DO MODELO =====
    mlflow.log_params({
        'model_type': 'XGBClassifier',
        'n_estimators': 100,
        'max_depth': 6,
        'learning_rate': 0.1,
        'random_state': 42,
        'eval_metric': 'logloss',
        'stratified_split': True,
        'categorical_features': str(cat_cols),
        'categorical_features_count': len(cat_cols),
        'numeric_features_count': len(num_cols),
        'total_features_encoded': X_train.shape[1],
        'train_size': len(X_train),
        'test_size': len(X_test),
        'train_test_ratio': round(len(X_train) / len(X_test), 2)
    })

    # ===== MÉTRICAS DE DESEMPENHO =====
    mlflow.log_metrics({
        'auc_roc': float(auc_roc),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1),
        'accuracy': float((y_pred == y_test).sum() / len(y_test)),
        'true_negatives': int(((y_pred == 0) & (y_test == 0)).sum()),
        'true_positives': int(((y_pred == 1) & (y_test == 1)).sum()),
        'false_negatives': int(((y_pred == 0) & (y_test == 1)).sum()),
        'false_positives': int(((y_pred == 1) & (y_test == 0)).sum())
    })

    # ===== ARTEFATOS: MODELO =====
    # Salvar modelo XGBoost
    model_path = os.path.join(tempfile.gettempdir(), 'xgboost_model.pkl')
    joblib.dump(model, model_path)
    mlflow.log_artifact(model_path, 'model')

    # Salvar encoder (necessário para inferência em produção)
    encoder_path = os.path.join(tempfile.gettempdir(), 'onehot_encoder.pkl')
    joblib.dump(encoder, encoder_path)
    mlflow.log_artifact(encoder_path, 'preprocessing')

    # ===== ARTEFATOS: AVALIAÇÃO =====
    # Curva ROC-AUC
    fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
    RocCurveDisplay.from_predictions(y_test, y_pred_proba, ax=ax_roc, name='XGBoost SLA Risk')
    ax_roc.set_title(f'ROC Curve — AUC-ROC: {auc_roc:.4f}')
    ax_roc.grid(True, alpha=0.3)
    roc_path = os.path.join(tempfile.gettempdir(), 'roc_curve.png')
    fig_roc.savefig(roc_path, dpi=100, bbox_inches='tight')
    mlflow.log_artifact(roc_path, 'evaluation')
    plt.close(fig_roc)

    # Confusion Matrix
    fig_cm, ax_cm = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax_cm, cmap='Blues', values_format='d')
    ax_cm.set_title(f'Confusion Matrix — Accuracy: {(y_pred == y_test).sum() / len(y_test):.4f}')
    cm_path = os.path.join(tempfile.gettempdir(), 'confusion_matrix.png')
    fig_cm.savefig(cm_path, dpi=100, bbox_inches='tight')
    mlflow.log_artifact(cm_path, 'evaluation')
    plt.close(fig_cm)

    # Distribuição de confiança (y_pred_proba)
    fig_dist, ax_dist = plt.subplots(figsize=(10, 5))
    ax_dist.hist(y_pred_proba[y_test == 0], bins=30, alpha=0.6, label='SLA OK (actual)', color='green')
    ax_dist.hist(y_pred_proba[y_test == 1], bins=30, alpha=0.6, label='SLA VIOLADO (actual)', color='red')
    ax_dist.axvline(0.5, color='black', linestyle='--', linewidth=2, label='Decision Threshold')
    ax_dist.set_xlabel('Predicted Probability of SLA Violation')
    ax_dist.set_ylabel('Frequency')
    ax_dist.set_title('Distribution of Predicted Probabilities')
    ax_dist.legend()
    ax_dist.grid(True, alpha=0.3)
    dist_path = os.path.join(tempfile.gettempdir(), 'confidence_distribution.png')
    fig_dist.savefig(dist_path, dpi=100, bbox_inches='tight')
    mlflow.log_artifact(dist_path, 'evaluation')
    plt.close(fig_dist)

    # ===== METADADOS E TAGS =====
    mlflow.set_tag('model_type', 'XGBoost')
    mlflow.set_tag('task', 'Binary_Classification_SLA_Risk')
    mlflow.set_tag('target_variable', 'target_excedeu_tempo')
    mlflow.set_tag('target_distribution', '80.3% class 0 (OK) / 19.7% class 1 (VIOLADO)')
    mlflow.set_tag('data_version', '2026-05-20_gold_ml_dataset')
    mlflow.set_tag('notebook', '06_model_risk_xgboost_shap')
    mlflow.set_tag('evaluation_metric', 'AUC-ROC')
    mlflow.set_tag('data_leakage_prevention', 'stratified_split + fit_encoder_on_train_only')
    mlflow.set_tag('feature_engineering', 'OneHotEncoding_4_categorical + 11_numeric')

    # ===== ASSINATURA DO MODELO (para serving) =====
    signature = infer_signature(X_test, y_pred_proba)
    mlflow.sklearn.log_model(model, 'model_with_signature', signature=signature)

    print('✅ MLflow tracking completo realizado')
    print(f'   Run name: xgboost_v1_sla_risk_classification')
    print(f'   Métricas: AUC-ROC={auc_roc:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}')
    print(f'   Artefatos: modelo (XGBoost), encoder (OneHotEncoder), plots (ROC, CM, distribuição)')
    print(f'   Tags: model_type, task, target_variable, data_version, notebook, evaluation_metric')

✅ MLflow tracking completo realizado
   Run name: xgboost_v1_sla_risk_classification
   Métricas: AUC-ROC=0.9710, Precision=0.8624, Recall=0.8564, F1=0.8594
   Artefatos: modelo (XGBoost), encoder (OneHotEncoder), plots (ROC, CM, distribuição)
   Tags: model_type, task, target_variable, data_version, notebook, evaluation_metric


In [33]:
# FEATURE SELECTION — Reduzir de 598 para ~50 features mais importantes
print('🔍 Feature Selection baseado em SHAP importance...')
print(f'\nAnálise do SHAP importance (598 features):')
print(f'  - Features com SHAP > 0.01: {(feature_importance["shap_importance"] > 0.01).sum()}')
print(f'  - Features com SHAP > 0.001: {(feature_importance["shap_importance"] > 0.001).sum()}')
print(f'  - Features com SHAP = 0.0: {(feature_importance["shap_importance"] == 0.0).sum()}')

# Selecionar top features (threshold = 0.005 reduz para ~50-80 features)
threshold_shap = 0.005
top_features = feature_importance[feature_importance['shap_importance'] > threshold_shap]['feature'].tolist()

print(f'\n✂️ Selecionando features com SHAP > {threshold_shap}:')
print(f'  - Features selecionadas: {len(top_features)} (redução de {(1 - len(top_features)/598)*100:.1f}%)')
print(f'  - Top 10 features: {top_features[:10]}')

# Treinar modelo com features selecionadas
X_train_sel = X_train[top_features]
X_test_sel = X_test[top_features]

model_sel = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss')
print(f'\n🚀 Treinando XGBoost com {len(top_features)} features selecionadas...')
model_sel.fit(X_train_sel, y_train, verbose=0)

# Avaliar modelo reduzido
y_pred_proba_sel = model_sel.predict_proba(X_test_sel)[:, 1]
y_pred_sel = model_sel.predict(X_test_sel)
auc_roc_sel = roc_auc_score(y_test, y_pred_proba_sel)
precision_sel = precision_score(y_test, y_pred_sel)
recall_sel = recall_score(y_test, y_pred_sel)
f1_sel = f1_score(y_test, y_pred_sel)

print(f'\n📊 Comparação de Performance:')
print(f'  Model Completo (598 features):')
print(f'    - AUC-ROC: {auc_roc:.4f}')
print(f'  Model Reduzido ({len(top_features)} features):')
print(f'    - AUC-ROC: {auc_roc_sel:.4f}')
print(f'    - Diferença: {(auc_roc - auc_roc_sel)*100:.2f}% (loss in AUC)')
print(f'\n✅ Trade-off: {(1 - len(top_features)/598)*100:.1f}% redução de features com apenas {(auc_roc - auc_roc_sel)*100:.2f}% de perda em AUC')

# Log no MLflow (nested run para feature selection)
with mlflow.start_run(nested=True, run_name='feature_selection_shap'):
    mlflow.log_params({
        'shap_threshold': threshold_shap,
        'n_features_original': 598,
        'n_features_selected': len(top_features),
        'reduction_percentage': round((1 - len(top_features)/598)*100, 2)
    })
    
    mlflow.log_metrics({
        'auc_roc_original': float(auc_roc),
        'auc_roc_selected': float(auc_roc_sel),
        'auc_roc_loss': float(auc_roc - auc_roc_sel),
        'precision_selected': float(precision_sel),
        'recall_selected': float(recall_sel),
        'f1_score_selected': float(f1_sel)
    })
    
    print(f'✅ Feature selection metrics logadas em nested run no MLflow')

🔍 Feature Selection baseado em SHAP importance...

Análise do SHAP importance (598 features):
  - Features com SHAP > 0.01: 18
  - Features com SHAP > 0.001: 52
  - Features com SHAP = 0.0: 512

✂️ Selecionando features com SHAP > 0.005:
  - Features selecionadas: 27 (redução de 95.5%)
  - Top 10 features: ['prioridade_num', 'categoria_0', 'possui_pai', 'grupo_designado_Team14', 'mes_abertura', 'hora_abertura', 'semana_ano', 'subcategoria_0', 'turno_abertura_Manha', 'turno_abertura_Tarde']

🚀 Treinando XGBoost com 27 features selecionadas...

📊 Comparação de Performance:
  Model Completo (598 features):
    - AUC-ROC: 0.9710
  Model Reduzido (27 features):
    - AUC-ROC: 0.9708
    - Diferença: 0.02% (loss in AUC)

✅ Trade-off: 95.5% redução de features com apenas 0.02% de perda em AUC
✅ Feature selection metrics logadas em nested run no MLflow


In [ ]:
# HYPERPARAMETER TUNING — GridSearch com Nested Runs no MLflow
from itertools import product

print('🔍 GridSearch: Testando combinações de hiperparâmetros...')

param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [50, 100, 150]
}

# Usar features selecionadas para tuning (mais rápido)
X_train_tuning = X_train_sel
X_test_tuning = X_test_sel

results_tuning = []

# Iterar sobre combinações (parent run já aberta)
param_combinations = list(product(
    param_grid['max_depth'],
    param_grid['learning_rate'],
    param_grid['n_estimators']
))

print(f'Total de combinações: {len(param_combinations)}')

for i, (max_d, lr, n_est) in enumerate(param_combinations, 1):
    # Cada combinação é um nested run
    with mlflow.start_run(nested=True, run_name=f'xgb_tuning_d{max_d}_lr{lr}_n{n_est}'):
        # Log dos parâmetros
        mlflow.log_params({
            'max_depth': max_d,
            'learning_rate': lr,
            'n_estimators': n_est
        })
        
        # Treinar modelo com essa combinação
        model_tune = XGBClassifier(
            n_estimators=n_est,
            max_depth=max_d,
            learning_rate=lr,
            random_state=42,
            eval_metric='logloss'
        )
        model_tune.fit(X_train_tuning, y_train, verbose=0)
        
        # Avaliar
        y_pred_proba_tune = model_tune.predict_proba(X_test_tuning)[:, 1]
        y_pred_tune = model_tune.predict(X_test_tuning)
        auc_tune = roc_auc_score(y_test, y_pred_proba_tune)
        f1_tune = f1_score(y_test, y_pred_tune)
        
        # Log das métricas
        mlflow.log_metrics({
            'auc_roc': float(auc_tune),
            'f1_score': float(f1_tune),
            'train_size': len(X_train_tuning),
            'test_size': len(X_test_tuning)
        })
        
        results_tuning.append({
            'max_depth': max_d,
            'learning_rate': lr,
            'n_estimators': n_est,
            'auc_roc': auc_tune,
            'f1_score': f1_tune
        })
        
        if i % 3 == 0:
            print(f'  [{i}/{len(param_combinations)}] max_depth={max_d}, lr={lr}, n_est={n_est} → AUC={auc_tune:.4f}')

# Encontrar melhor combinação
df_tuning = pd.DataFrame(results_tuning)
best_idx = df_tuning['auc_roc'].idxmax()
best_params = df_tuning.loc[best_idx]

print(f'\n🏆 Melhor combinação encontrada:')
print(f'  - max_depth: {best_params["max_depth"]:.0f}')
print(f'  - learning_rate: {best_params["learning_rate"]}')
print(f'  - n_estimators: {best_params["n_estimators"]:.0f}')
print(f'  - AUC-ROC: {best_params["auc_roc"]:.4f}')
print(f'  - F1-Score: {best_params["f1_score"]:.4f}')

# Visualizar grid de tuning
fig_grid, ax = plt.subplots(figsize=(12, 6))

# Criar matrix de AUC por max_depth e learning_rate
pivot_table = df_tuning.pivot_table(
    values='auc_roc',
    index='max_depth',
    columns='learning_rate',
    aggfunc='max'  # Se houver múltiplos n_estimators, pega o melhor
)

import seaborn as sns
sns.heatmap(pivot_table, annot=True, fmt='.4f', cmap='YlGn', ax=ax, cbar_kws={'label': 'AUC-ROC'})
ax.set_title('GridSearch: AUC-ROC por max_depth e learning_rate (n_estimators variável)')
ax.set_xlabel('Learning Rate')
ax.set_ylabel('Max Depth')

grid_path = os.path.join(tempfile.gettempdir(), 'gridsearch_heatmap.png')
fig_grid.savefig(grid_path, dpi=100, bbox_inches='tight')
plt.close(fig_grid)

# Log da figura no MLflow parent run
mlflow.log_artifact(grid_path, 'hyperparameter_tuning')
mlflow.log_metrics({
    'best_auc_roc': float(best_params['auc_roc']),
    'best_f1_score': float(best_params['f1_score']),
    'tuning_combinations': len(param_combinations)
})

print(f'\n✅ GridSearch completo: {len(param_combinations)} modelos treinados')
print(f'   Melhoria em relação ao baseline: {(best_params["auc_roc"] - auc_roc_sel)*100:.2f}% em AUC')

## 📁 Etapa 8: Salvamento de Resultados e Artefatos

Agora vou exportar **predições, importâncias e métricas** em CSV para uso em dashboards e análises downstream.

**Arquivos gerados em `/data/ml/xgboost/`**:

| Arquivo | Conteúdo | Uso |
|---------|----------|-----|
| **xgboost_test_predictions.csv** | Para cada incidente do teste: y_real (SLA real), y_pred (previsão 0/1), y_pred_proba (confiança 0-1) | Power BI: filtrar "incidentes em risco" para alertas |
| **xgboost_shap_importance.csv** | Feature name + SHAP importance (média de valores absolutos) | Entender quais características mais influenciam risco |
| **xgboost_summary.csv** | AUC-ROC, #features, tamanho train/test | Auditoria: metadados quando o modelo foi executado |

**Próximas ações**:
1. **Power BI**: Conectar `xgboost_test_predictions.csv` para criar painel "Incidentes em Risco"
2. **Alertas**: Criar trigger automático para incidentes com `y_pred_proba > 0.8`
3. **Análise SHAP**: Estudar por que certos grupos/categorias têm maior risco

## 📊 Etapa 7: Rastreamento com MLflow

Agora vou registrar o modelo, métricas e artefatos no **MLflow** para rastreabilidade completa.

### Por que rastrear com MLflow?
- **Reprodutibilidade**: Futuros refinamentos precisam saber exatamente qual configuração foi usada
- **Governança**: Quem treinou, quando, com quais hiperparâmetros
- **Auditoria**: Conformidade — "qual foi a AUC-ROC quando esse modelo entrou em produção?"
- **Comparação**: Testar max_depth=4 vs. 6 lado-a-lado no MLflow UI
- **Deploy**: Versão marcada como "produção" fica acessível para serving

### O que será registrado:
- **Parâmetros**: n_estimators=100, max_depth=6, learning_rate=0.1
- **Métricas**: AUC-ROC, Precision, Recall, F1-Score
- **Artefatos**: Modelo XGBoost serializado (joblib)
- **Metadados**: Tamanho train/test, número de features

In [ ]:
# Salvar resultados em data/ml/xgboost/
from pathlib import Path

base_path = Path(r'D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\xgboost')
base_path.mkdir(parents=True, exist_ok=True)

print(f'Salvando em: {base_path}')

# Salvar predições e métricas do teste
test_results = pd.DataFrame({
    'y_real': y_test.values,
    'y_pred': y_pred,
    'y_pred_proba': y_pred_proba
})
test_results.to_csv(str(base_path / 'xgboost_test_predictions.csv'), index=False)
print(f'✅ Saved: xgboost_test_predictions.csv')

# Salvar importância SHAP
feature_importance.to_csv(str(base_path / 'xgboost_shap_importance.csv'), index=False)
print(f'✅ Saved: xgboost_shap_importance.csv')

# Resumo de métricas
summary = pd.DataFrame({
    'métrica': ['AUC-ROC', 'Total features', 'Train size', 'Test size'],
    'valor': [f'{auc_roc:.4f}', len(feature_cols), len(X_train), len(X_test)]
})
summary.to_csv(str(base_path / 'xgboost_summary.csv'), index=False)
print(f'✅ Saved: xgboost_summary.csv')

Salvando em: D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\xgboost
✅ Saved: xgboost_test_predictions.csv
✅ Saved: xgboost_shap_importance.csv
✅ Saved: xgboost_summary.csv
